In [2]:
# =========================
# 1. Imports and setup
# =========================
from pathlib import Path
import sys
import os
import json
import re
import random
import warnings
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils import check_random_state


def resolve_project_root(start: Path) -> Path:
    """Find the project root that contains src/ and data/."""
    for p in [start, *start.parents]:
        if (p / "src").exists() and (p / "data").exists():
            return p
        nested = p / "document-classification"
        if (nested / "src").exists() and (nested / "data").exists():
            return nested
    raise RuntimeError(
        "Could not locate project root with src/ and data/. "
        "Run this notebook from inside the repository workspace."
    )


PROJECT_ROOT = resolve_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings("ignore")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
REPORTS_DIR = PROJECT_ROOT / "reports"

PREDICTIONS_DIR = OUTPUTS_DIR / "predictions"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"

for d in [MODELS_DIR, PREDICTIONS_DIR, FIGURES_DIR, TABLES_DIR, INTERIM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# -------------------------
# Text feature helpers
# -------------------------
try:
    from src.text_features import (
        clean_ocr_text,
        build_word_tfidf_vectorizer,
        build_char_tfidf_vectorizer,
        build_combined_word_char_vectorizer,
    )
except Exception:
    from scipy.sparse import hstack
    from sklearn.base import BaseEstimator, TransformerMixin
    from sklearn.feature_extraction.text import TfidfVectorizer

    _WS_RE = re.compile(r"\s+")
    _CTRL_RE = re.compile(r"[\x00-\x08\x0B-\x1F\x7F]")
    _BROKEN_RE = re.compile(r"[^\w\s\.,:;/%$€£¥#@&\-\(\)/]")

    def _normalize_whitespace(text: str) -> str:
        return _WS_RE.sub(" ", str(text)).strip()

    def clean_ocr_text(text: Optional[str]) -> str:
        if text is None:
            return ""
        t = str(text).lower()
        t = _CTRL_RE.sub(" ", t)
        t = _BROKEN_RE.sub(" ", t)
        t = re.sub(r"([:;/,\.\-]){2,}", r"\1", t)
        t = re.sub(r"\b[a-z]{1}\b", " ", t)
        return _normalize_whitespace(t)

    def build_word_tfidf_vectorizer(
        ngram_range: Tuple[int, int] = (1, 2),
        max_features: int = 30000,
        min_df: int = 2,
        max_df: float = 0.98,
    ):
        return TfidfVectorizer(
            analyzer="word",
            ngram_range=ngram_range,
            max_features=max_features,
            min_df=min_df,
            max_df=max_df,
            strip_accents="unicode",
            sublinear_tf=True,
        )

    def build_char_tfidf_vectorizer(
        ngram_range: Tuple[int, int] = (3, 5),
        max_features: int = 30000,
        min_df: int = 2,
        max_df: float = 1.0,
    ):
        return TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=ngram_range,
            max_features=max_features,
            min_df=min_df,
            max_df=max_df,
            sublinear_tf=True,
        )

    class _CombinedWordCharTfidfVectorizer(BaseEstimator, TransformerMixin):
        def __init__(
            self,
            word_ngram_range: Tuple[int, int] = (1, 2),
            char_ngram_range: Tuple[int, int] = (3, 5),
            max_word_features: int = 30000,
            max_char_features: int = 30000,
            min_df: int = 2,
        ) -> None:
            self.word_vectorizer = build_word_tfidf_vectorizer(
                ngram_range=word_ngram_range,
                max_features=max_word_features,
                min_df=min_df,
            )
            self.char_vectorizer = build_char_tfidf_vectorizer(
                ngram_range=char_ngram_range,
                max_features=max_char_features,
                min_df=min_df,
            )

        def fit(self, X, y=None):
            self.word_vectorizer.fit(X)
            self.char_vectorizer.fit(X)
            return self

        def transform(self, X):
            return hstack([
                self.word_vectorizer.transform(X),
                self.char_vectorizer.transform(X),
            ]).tocsr()

        def get_feature_names_out(self):
            w = self.word_vectorizer.get_feature_names_out()
            c = self.char_vectorizer.get_feature_names_out()
            return np.concatenate([w, c])

    def build_combined_word_char_vectorizer(
        word_ngram_range: Tuple[int, int] = (1, 2),
        char_ngram_range: Tuple[int, int] = (3, 5),
        max_word_features: int = 30000,
        max_char_features: int = 30000,
        min_df: int = 2,
    ):
        return _CombinedWordCharTfidfVectorizer(
            word_ngram_range=word_ngram_range,
            char_ngram_range=char_ngram_range,
            max_word_features=max_word_features,
            max_char_features=max_char_features,
            min_df=min_df,
        )

# -------------------------
# Baseline model helpers
# -------------------------
try:
    from src.models_baseline import (
        train_logistic_regression_baseline,
        train_linear_svm_baseline,
        evaluate_classifier,
        select_best_model_from_validation,
        extract_top_features_per_class,
        save_model_bundle,
    )
except Exception:
    import joblib
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
    from sklearn.svm import LinearSVC

    def train_logistic_regression_baseline(X_train, y_train, C: float = 1.0, random_state: int = 42):
        model = LogisticRegression(C=C, max_iter=4000, multi_class="ovr", solver="liblinear", random_state=random_state)
        model.fit(X_train, y_train)
        return model

    def train_linear_svm_baseline(X_train, y_train, C: float = 1.0, random_state: int = 42):
        model = LinearSVC(C=C, random_state=random_state)
        model.fit(X_train, y_train)
        return model

    def _compute_metrics(y_true, y_pred, labels: List[str]) -> Dict[str, Any]:
        acc = accuracy_score(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average="macro")
        weighted_f1 = f1_score(y_true, y_pred, average="weighted")
        p, r, f1, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
        per_class = {}
        for cls, pp, rr, ff, ss in zip(labels, p, r, f1, s):
            per_class[cls] = {"precision": float(pp), "recall": float(rr), "f1": float(ff), "support": int(ss)}
        invoice_recall = per_class["invoice"]["recall"] if "invoice" in per_class else None
        return {"accuracy": float(acc), "macro_f1": float(macro_f1), "weighted_f1": float(weighted_f1), "invoice_recall": invoice_recall, "per_class": per_class}

    def evaluate_classifier(model, X, y_true, label_order: List[str]):
        y_pred = model.predict(X)
        return _compute_metrics(y_true, y_pred, labels=label_order), y_pred

    def select_best_model_from_validation(candidate_runs: List[Dict[str, Any]], primary_metric: str = "macro_f1") -> Dict[str, Any]:
        best = None
        best_score = -np.inf
        for run in candidate_runs:
            score = run["val_metrics"][primary_metric]
            if score > best_score:
                best_score = score
                best = run
            elif score == best_score and best is not None:
                current_ir = run["val_metrics"].get("invoice_recall", -1)
                best_ir = best["val_metrics"].get("invoice_recall", -1)
                if current_ir > best_ir:
                    best = run
                elif current_ir == best_ir:
                    current_acc = run["val_metrics"].get("accuracy", -1)
                    best_acc = best["val_metrics"].get("accuracy", -1)
                    if current_acc > best_acc:
                        best = run
        if best is None:
            raise ValueError("No candidate runs were provided.")
        return best

    def extract_top_features_per_class(model, vectorizer, top_k: int = 20) -> pd.DataFrame:
        if not hasattr(model, "coef_"):
            raise ValueError("Model does not expose linear coefficients.")
        feature_names = np.array(vectorizer.get_feature_names_out())
        classes = list(model.classes_)
        rows = []
        for class_idx, cls in enumerate(classes):
            coef = model.coef_[class_idx]
            top_pos_idx = np.argsort(coef)[-top_k:][::-1]
            for rank, feat_idx in enumerate(top_pos_idx, start=1):
                rows.append({"class_name": cls, "rank": rank, "feature": feature_names[feat_idx], "weight": float(coef[feat_idx])})
        return pd.DataFrame(rows)

    def save_model_bundle(save_path: Path, model, vectorizer, metadata: Dict[str, Any]) -> None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        joblib.dump({"model": model, "vectorizer": vectorizer, "metadata": metadata}, save_path)

# -------------------------
# OCR compatibility helpers
# -------------------------
from src.config import OCRConfig, load_ocr_config
from src.ocr_engine import ocr_batch, load_ocr_text, check_tesseract_installation


def tesseract_is_available() -> bool:
    try:
        return bool(check_tesseract_installation(verbose=False))
    except TypeError:
        return bool(check_tesseract_installation())


def load_cached_ocr_for_split(split_name: str, cache_dir: Path) -> Optional[pd.DataFrame]:
    cache_csv = Path(cache_dir) / f"{split_name}_ocr_cache.csv"
    return pd.read_csv(cache_csv) if cache_csv.exists() else None


def ensure_ocr_for_split(df: pd.DataFrame, split_name: str, cache_dir: Path, force_recompute: bool = False) -> pd.DataFrame:
    out = df.copy()
    if not {"doc_id", "file_path", "class_name"}.issubset(out.columns):
        raise ValueError("Expected columns: doc_id, file_path, class_name")

    cfg_path = PROJECT_ROOT / "configs" / "config.yaml"
    cfg = load_ocr_config(cfg_path) if cfg_path.exists() else OCRConfig()
    cfg.cache_dir = str(Path(cache_dir))
    cfg.diagnostics_dir = str(OUTPUTS_DIR / "ocr_diagnostics")
    cfg.failure_log_path = str(Path(cache_dir) / "logs" / "ocr_failures.jsonl")

    if "split" not in out.columns:
        out["split"] = split_name
    else:
        out["split"] = out["split"].fillna(split_name)

    ocr_batch(
        metadata_df=out[["doc_id", "file_path", "class_name", "split"]],
        cfg=cfg,
        force=force_recompute,
        show_progress=True,
    )

    out["ocr_text"] = out["doc_id"].map(lambda d: load_ocr_text(str(d), cfg=cfg))

    cache_csv = Path(cache_dir) / f"{split_name}_ocr_cache.csv"
    cache_csv.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(cache_csv, index=False)
    return out

# -------------------------
# Evaluation helpers
# -------------------------
try:
    from src.evaluation import (
        plot_confusion_matrix_matplotlib,
        save_classification_report_table,
        save_metrics_json,
    )
except Exception:
    def plot_confusion_matrix_matplotlib(y_true, y_pred, labels, title, save_path):
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        fig, ax = plt.subplots(figsize=(8, 8))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
        disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
        ax.set_title(title)
        fig.tight_layout()
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
        plt.close(fig)

    def save_classification_report_table(y_true, y_pred, save_path, labels):
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        report = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
        df = pd.DataFrame(report).T
        df.to_csv(save_path)
        return df

    def save_metrics_json(metrics, save_path):
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2)

print("Working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Tesseract available:", tesseract_is_available())



Working directory: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/notebooks
Project root: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification
Tesseract available: True


## Dataset label verification

Before training the baseline, we verify the dataset structure programmatically instead of hardcoding assumptions.

There is a known project-description mismatch between:
- `invoice, receipts, resume, email, budget`
and
- `invoice, form, resume, email, budget`

This notebook inspects the available labels from the downloaded dataset and explicitly determines whether the intended class is `receipts` or `form`.

In [3]:
# =========================
# 2. Dataset label verification
# =========================
def discover_candidate_labels(raw_dir: Path) -> Dict[str, List[Path]]:
    """
    Search recursively for label-like folders.
    Returns a dict mapping folder name -> list of matching folder paths.
    """
    label_map = {}
    if not raw_dir.exists():
        print(f"[WARN] Raw directory does not exist yet: {raw_dir}")
        return label_map

    for p in raw_dir.rglob("*"):
        if p.is_dir():
            folder_name = p.name.strip().lower()
            if folder_name not in label_map:
                label_map[folder_name] = []
            label_map[folder_name].append(p)

    return label_map

label_map = discover_candidate_labels(RAW_DIR)

interesting_names = sorted([
    k for k in label_map.keys()
    if k in {
        "invoice", "invoices",
        "receipt", "receipts",
        "form", "forms",
        "resume", "resumes",
        "email", "emails",
        "budget", "budgets"
    }
])

print("Detected relevant candidate folder names:")
for name in interesting_names:
    print(f"  - {name}: {len(label_map[name])} path(s)")
    for p in label_map[name][:5]:
        print("      ", p)

has_form = any(k in label_map for k in ["form", "forms"])
has_receipts = any(k in label_map for k in ["receipt", "receipts"])

if has_form and not has_receipts:
    VERIFIED_LABEL_2 = "form"
elif has_receipts and not has_form:
    VERIFIED_LABEL_2 = "receipts"
elif has_form and has_receipts:
    VERIFIED_LABEL_2 = "form"  # default to official RVL-CDIP interpretation, but keep both visible
    print("[WARN] Both form and receipts-like folders were found. Defaulting to 'form' for baseline unless metadata proves otherwise.")
else:
    VERIFIED_LABEL_2 = None

TARGET_CLASSES = ["invoice", VERIFIED_LABEL_2, "resume", "email", "budget"] if VERIFIED_LABEL_2 else None
print("\nVerified class mapping:")
print("  invoice -> invoice")
print(f"  second class -> {VERIFIED_LABEL_2}")
print("  resume -> resume")
print("  email -> email")
print("  budget -> budget")

if TARGET_CLASSES is None or any(x is None for x in TARGET_CLASSES):
    raise RuntimeError(
        "Could not verify whether the target second class is 'form' or 'receipts'. "
        "Inspect the dataset structure under data/raw/ and update the mapping."
    )

Detected relevant candidate folder names:
  - budget: 1 path(s)
       /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/raw/kaggle_extracted/test/budget
  - email: 1 path(s)
       /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/raw/kaggle_extracted/test/email
  - form: 1 path(s)
       /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/raw/kaggle_extracted/test/form
  - invoice: 1 path(s)
       /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/raw/kaggle_extracted/test/invoice
  - resume: 1 path(s)
       /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/raw/kaggle_extracted/test/resume

Verified class mapping:
  invoice -> invoice
  second class -> form
  resume -> resume
  email -> email
  budget -> budget


## Split loading and fallback protocol

The preferred workflow is to reuse the canonical group split files:

- `data/processed/train.csv`
- `data/processed/val.csv`
- `data/processed/test.csv`

If they are unavailable, this notebook creates a clean fallback split from the verified five-class subset only, using stratified splitting and a fixed random seed.

To avoid data leakage:
- the test split is never used during model selection
- vectorizers are fit on training data only
- validation is used for model selection
- final test is reported once for the selected best model

In [4]:
# =========================
# 3. Load canonical splits or create fallback
# =========================
TRAIN_CSV = PROCESSED_DIR / "train.csv"
VAL_CSV = PROCESSED_DIR / "val.csv"
TEST_CSV = PROCESSED_DIR / "test.csv"
META_CSV = PROCESSED_DIR / "metadata_five_classes.csv"

def load_existing_splits():
    if TRAIN_CSV.exists() and VAL_CSV.exists() and TEST_CSV.exists():
        train_df = pd.read_csv(TRAIN_CSV)
        val_df = pd.read_csv(VAL_CSV)
        test_df = pd.read_csv(TEST_CSV)
        return train_df, val_df, test_df
    return None, None, None

def infer_label_from_path(path_str: str, target_classes: List[str]) -> Optional[str]:
    p = str(path_str).lower()
    for cls in target_classes:
        if cls and cls in p:
            return cls
    return None

def build_fallback_metadata_from_raw(raw_dir: Path, target_classes: List[str]) -> pd.DataFrame:
    image_exts = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
    rows = []

    for fp in raw_dir.rglob("*"):
        if fp.is_file() and fp.suffix.lower() in image_exts:
            label = infer_label_from_path(str(fp), target_classes)
            if label is None:
                continue
            rows.append({
                "doc_id": fp.stem,
                "file_path": str(fp.resolve()),
                "class_name": label,
                "source_folder": fp.parent.name,
                "file_ext": fp.suffix.lower()
            })

    meta = pd.DataFrame(rows).drop_duplicates(subset=["file_path"]).reset_index(drop=True)
    return meta

def create_stratified_splits(meta: pd.DataFrame, seed: int = 42):
    train_val_df, test_df = train_test_split(
        meta,
        test_size=0.15,
        random_state=seed,
        stratify=meta["class_name"]
    )

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=0.1764705882,  # gives roughly 15% val after holding out 15% test
        random_state=seed,
        stratify=train_val_df["class_name"]
    )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

train_df, val_df, test_df = load_existing_splits()

if train_df is None:
    print("[INFO] Canonical split CSVs not found. Building fallback splits.")
    if META_CSV.exists():
        meta_df = pd.read_csv(META_CSV)
        meta_df = meta_df[meta_df["class_name"].isin(TARGET_CLASSES)].copy()
    else:
        meta_df = build_fallback_metadata_from_raw(RAW_DIR, TARGET_CLASSES)

    if meta_df.empty:
        raise RuntimeError("No fallback metadata could be built from data/raw/. Please download and unpack the dataset first.")

    train_df, val_df, test_df = create_stratified_splits(meta_df, seed=SEED)

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(TRAIN_CSV, index=False)
    val_df.to_csv(VAL_CSV, index=False)
    test_df.to_csv(TEST_CSV, index=False)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{name.upper()} class distribution")
    print(df["class_name"].value_counts())

Train shape: (8777, 12)
Val shape: (1882, 12)
Test shape: (1881, 12)

TRAIN class distribution
class_name
resume     1775
email      1761
form       1754
budget     1753
invoice    1734
Name: count, dtype: int64

VAL class distribution
class_name
resume     381
email      378
form       376
budget     376
invoice    371
Name: count, dtype: int64

TEST class distribution
class_name
resume     380
email      377
form       376
budget     376
invoice    372
Name: count, dtype: int64


In [5]:
# =========================
# 4. Sanity checks
# =========================
def verify_split_dataframe(df: pd.DataFrame, split_name: str):
    required_cols = {"doc_id", "file_path", "class_name"}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"{split_name} is missing required columns: {missing_cols}")

    missing_files = []
    invalid_labels = []

    for _, row in df.iterrows():
        fp = Path(row["file_path"])
        if not fp.exists():
            missing_files.append(str(fp))
        if row["class_name"] not in TARGET_CLASSES:
            invalid_labels.append(row["class_name"])

    print(f"[{split_name}] missing files: {len(missing_files)}")
    print(f"[{split_name}] invalid labels: {len(invalid_labels)}")

    if missing_files[:5]:
        print("Sample missing files:")
        print("\n".join(missing_files[:5]))

    if invalid_labels[:5]:
        print("Sample invalid labels:")
        print(invalid_labels[:5])

for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    verify_split_dataframe(df, split_name)

[train] missing files: 0
[train] invalid labels: 0
[val] missing files: 0
[val] invalid labels: 0
[test] missing files: 0
[test] invalid labels: 0


## OCR loading / generation

This baseline depends entirely on OCR text, so OCR quality directly impacts classification quality.

The pipeline:
- loads cached OCR outputs if available
- otherwise runs Tesseract OCR
- stores OCR outputs under `data/interim/ocr/`
- preserves a one-to-one mapping between `doc_id` and extracted text

This keeps the workflow reproducible and efficient.

In [6]:
# =========================
# 5. OCR cache loading or generation
# =========================
OCR_CACHE_DIR = INTERIM_DIR / "ocr"
OCR_CACHE_DIR.mkdir(parents=True, exist_ok=True)

train_df = ensure_ocr_for_split(train_df, split_name="train", cache_dir=OCR_CACHE_DIR)
val_df = ensure_ocr_for_split(val_df, split_name="val", cache_dir=OCR_CACHE_DIR)
test_df = ensure_ocr_for_split(test_df, split_name="test", cache_dir=OCR_CACHE_DIR)

for df in [train_df, val_df, test_df]:
    if "ocr_text" not in df.columns:
        raise RuntimeError("OCR text was not attached to the split dataframe.")

OCR batch: 100%|██████████| 1881/1881 [00:23<00:00, 79.86it/s]


## Text preprocessing

The text normalization strategy is intentionally conservative.

We:
- lowercase text
- normalize whitespace
- remove obvious OCR junk carefully
- preserve useful symbols when beneficial
- preserve dates and currency cues where possible

This is important because invoice-like documents often depend on strings such as:
- `invoice`
- `date`
- `amount`
- currency markers
- IDs
- reference numbers

Over-cleaning would destroy useful signal.

In [7]:
# =========================
# 6. Text preprocessing
# =========================
train_df["text_clean"] = train_df["ocr_text"].fillna("").apply(clean_ocr_text)
val_df["text_clean"] = val_df["ocr_text"].fillna("").apply(clean_ocr_text)
test_df["text_clean"] = test_df["ocr_text"].fillna("").apply(clean_ocr_text)

print(train_df[["doc_id", "class_name", "text_clean"]].head(3))

         doc_id class_name                                         text_clean
0  doc_00000000       form  poa on zaphs ykennie wonaray usa. philip morri...
1  doc_00000001       form  ot oon - sah wed . ; cnr narave tutor cows 4 g...
2  doc_00000002       form  . surtgog . - . 5093 . , sersene mall crc cont...


## Feature engineering

We build TF-IDF features from:
- word n-grams
- character n-grams

The baseline uses both because:

- word n-grams capture semantic document cues such as headers and section names
- character n-grams are robust to OCR noise, spelling corruption, partial tokens, and formatting irregularities

This often makes word+character TF-IDF stronger than word-only text features in OCR-heavy document pipelines.

In [8]:
# =========================
# 7. Build feature representation
# =========================
X_train_text = train_df["text_clean"].tolist()
y_train = train_df["class_name"].tolist()

X_val_text = val_df["text_clean"].tolist()
y_val = val_df["class_name"].tolist()

X_test_text = test_df["text_clean"].tolist()
y_test = test_df["class_name"].tolist()

vectorizer = build_combined_word_char_vectorizer(
    word_ngram_range=(1, 2),
    char_ngram_range=(3, 5),
    max_word_features=30000,
    max_char_features=30000,
    min_df=2
)

X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(X_test_text)

print("Train feature matrix:", X_train.shape)
print("Val feature matrix:", X_val.shape)
print("Test feature matrix:", X_test.shape)

Train feature matrix: (8777, 60000)
Val feature matrix: (1882, 60000)
Test feature matrix: (1881, 60000)


## Baseline models

We train two classical baselines:

1. Logistic Regression
2. Linear SVM

Model selection is based on validation performance only.  
The final test set is used only once after the best model is selected.

Primary selection metric:
- macro F1

Also monitored:
- overall accuracy
- weighted F1
- invoice recall

In [9]:
# =========================
# 8. Train baseline models
# =========================
label_order = sorted(train_df["class_name"].unique())

svm_grid = {
    "C": [0.25, 0.5, 1.0, 2.0]
}

logreg_grid = {
    "C": [0.25, 0.5, 1.0, 2.0]
}

svm_results = []
for params in ParameterGrid(svm_grid):
    model = train_linear_svm_baseline(
        X_train=X_train,
        y_train=y_train,
        C=params["C"],
        random_state=SEED
    )
    metrics, preds = evaluate_classifier(model, X_val, y_val, label_order=label_order)
    svm_results.append({
        "model_name": "linear_svm",
        "params": params,
        "model": model,
        "val_metrics": metrics,
        "val_preds": preds,
    })

logreg_results = []
for params in ParameterGrid(logreg_grid):
    model = train_logistic_regression_baseline(
        X_train=X_train,
        y_train=y_train,
        C=params["C"],
        random_state=SEED
    )
    metrics, preds = evaluate_classifier(model, X_val, y_val, label_order=label_order)
    logreg_results.append({
        "model_name": "logistic_regression",
        "params": params,
        "model": model,
        "val_metrics": metrics,
        "val_preds": preds,
    })

all_candidates = svm_results + logreg_results
best_run = select_best_model_from_validation(all_candidates, primary_metric="macro_f1")

print("Best validation model:")
print("  model:", best_run["model_name"])
print("  params:", best_run["params"])
print("  val metrics:", best_run["val_metrics"])

Best validation model:
  model: linear_svm
  params: {'C': 0.25}
  val metrics: {'accuracy': 0.8783209351753454, 'macro_f1': 0.8789548003502766, 'weighted_f1': 0.8795286612607365, 'invoice_recall': 0.8382749326145552, 'per_class': {'budget': {'precision': 0.8042328042328042, 'recall': 0.8085106382978723, 'f1': 0.8063660477453581, 'support': 376}, 'email': {'precision': 0.967479674796748, 'recall': 0.9444444444444444, 'f1': 0.9558232931726908, 'support': 378}, 'form': {'precision': 0.8764044943820225, 'recall': 0.8297872340425532, 'f1': 0.8524590163934426, 'support': 376}, 'invoice': {'precision': 0.7603911980440098, 'recall': 0.8382749326145552, 'f1': 0.7974358974358975, 'support': 371}, 'resume': {'precision': 0.9972972972972973, 'recall': 0.968503937007874, 'f1': 0.9826897470039947, 'support': 381}}}


In [10]:
# =========================
# 9. Validation evaluation and save validation predictions
# =========================
best_model = best_run["model"]
val_metrics, val_preds = evaluate_classifier(best_model, X_val, y_val, label_order=label_order)

val_pred_df = val_df.copy()
val_pred_df["y_true"] = y_val
val_pred_df["y_pred"] = val_preds

val_pred_path = PREDICTIONS_DIR / "baseline_val_predictions.csv"
val_pred_df.to_csv(val_pred_path, index=False)

val_report_path = TABLES_DIR / "baseline_val_classification_report.csv"
save_classification_report_table(y_val, val_preds, val_report_path, label_order)

val_cm_path = FIGURES_DIR / "baseline_val_confusion_matrix.png"
plot_confusion_matrix_matplotlib(
    y_true=y_val,
    y_pred=val_preds,
    labels=label_order,
    title="Validation Confusion Matrix — OCR TF-IDF Baseline",
    save_path=val_cm_path
)

val_metrics_path = TABLES_DIR / "baseline_val_metrics.json"
save_metrics_json(val_metrics, val_metrics_path)

print("Validation artifacts saved:")
print(val_pred_path)
print(val_report_path)
print(val_cm_path)
print(val_metrics_path)

Validation artifacts saved:
/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/outputs/predictions/baseline_val_predictions.csv
/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/reports/tables/baseline_val_classification_report.csv
/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/reports/figures/baseline_val_confusion_matrix.png
/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/reports/tables/baseline_val_metrics.json


## Final test evaluation

After selecting the best model using validation only, we evaluate once on the held-out test set.

A particularly important metric is **invoice recall**, because invoice detection determines which documents will later be passed to the downstream invoice extraction stage in the broader project.

In [11]:
# =========================
# 10. Final test evaluation
# =========================
test_metrics, test_preds = evaluate_classifier(best_model, X_test, y_test, label_order=label_order)

test_pred_df = test_df.copy()
test_pred_df["y_true"] = y_test
test_pred_df["y_pred"] = test_preds

test_pred_path = PREDICTIONS_DIR / "baseline_test_predictions.csv"
test_pred_df.to_csv(test_pred_path, index=False)

test_report_path = TABLES_DIR / "baseline_test_classification_report.csv"
save_classification_report_table(y_test, test_preds, test_report_path, label_order)

test_cm_path = FIGURES_DIR / "baseline_test_confusion_matrix.png"
plot_confusion_matrix_matplotlib(
    y_true=y_test,
    y_pred=test_preds,
    labels=label_order,
    title="Test Confusion Matrix — OCR TF-IDF Baseline",
    save_path=test_cm_path
)

test_metrics_path = TABLES_DIR / "baseline_test_metrics.json"
save_metrics_json(test_metrics, test_metrics_path)

print("Test metrics:", test_metrics)
print("Test artifacts saved:")
print(test_pred_path)
print(test_report_path)
print(test_cm_path)
print(test_metrics_path)

Test metrics: {'accuracy': 0.8766613503455609, 'macro_f1': 0.8774601351436464, 'weighted_f1': 0.8778263887890327, 'invoice_recall': 0.8387096774193549, 'per_class': {'budget': {'precision': 0.7684729064039408, 'recall': 0.8297872340425532, 'f1': 0.7979539641943734, 'support': 376}, 'email': {'precision': 0.964769647696477, 'recall': 0.9442970822281167, 'f1': 0.9544235924932976, 'support': 377}, 'form': {'precision': 0.8767908309455588, 'recall': 0.8138297872340425, 'f1': 0.8441379310344828, 'support': 376}, 'invoice': {'precision': 0.8, 'recall': 0.8387096774193549, 'f1': 0.8188976377952756, 'support': 372}, 'resume': {'precision': 0.989100817438692, 'recall': 0.9552631578947368, 'f1': 0.9718875502008032, 'support': 380}}}
Test artifacts saved:
/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/outputs/predictions/baseline_test_predictions.csv
/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/reports

## Feature interpretation

Linear models allow direct interpretation through feature weights.

This is useful because the baseline should not only be accurate, but also explainable:
- invoice indicators may include monetary and billing language
- form indicators may include field labels and structured prompts
- resume indicators may include education/experience terminology
- email indicators may include reply/subject/header conventions
- budget indicators may include totals, line items, and planning vocabulary

In [12]:
# =========================
# 11. Top discriminative features
# =========================
top_features_df = extract_top_features_per_class(
    model=best_model,
    vectorizer=vectorizer,
    top_k=20
)

top_features_path = TABLES_DIR / "baseline_top_features_per_class.csv"
top_features_df.to_csv(top_features_path, index=False)

display(top_features_df.head(50))
print("Saved:", top_features_path)

,class_name,rank,feature,weight
0,budget,1,00,1.044117
1,budget,2,tnwl,0.958519
2,budget,3,tobacco institute,0.933270
3,budget,4,timn,0.881531
4,budget,5,budget,0.880664
5,budget,6,cost,0.816636
6,budget,7,rr,0.702102
7,budget,8,000,0.689652
8,budget,9,tcal,0.680813
9,budget,10,the tobacco,0.676240


Saved: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/reports/tables/baseline_top_features_per_class.csv


## Error analysis

We inspect representative mistakes to understand where a text-only OCR baseline succeeds and fails.

Expected confusion pairs:
- invoice vs budget
- invoice vs form
- email vs resume

Typical causes of error may include:
- OCR corruption
- overlapping administrative vocabulary
- weak or missing headers
- visually distinctive layout that text alone cannot fully capture

In [13]:
# =========================
# 12. Error analysis
# =========================
def show_confusion_examples(df_pred: pd.DataFrame, true_label: str, pred_label: str, n: int = 5):
    subset = df_pred[(df_pred["y_true"] == true_label) & (df_pred["y_pred"] == pred_label)].head(n)
    print(f"\nExamples: true={true_label}, predicted={pred_label}, count shown={len(subset)}")
    for _, row in subset.iterrows():
        print("=" * 100)
        print("doc_id:", row["doc_id"])
        print("file_path:", row["file_path"])
        snippet = str(row.get("text_clean", ""))[:1000]
        print("OCR snippet:")
        print(snippet)

test_pred_df["text_clean"] = test_df["text_clean"].values

confusion_pairs = [
    ("invoice", VERIFIED_LABEL_2),
    (VERIFIED_LABEL_2, "invoice"),
    ("invoice", "budget"),
    ("budget", "invoice"),
    ("email", "resume"),
    ("resume", "email"),
]

for true_label, pred_label in confusion_pairs:
    show_confusion_examples(test_pred_df, true_label, pred_label, n=3)


Examples: true=invoice, predicted=form, count shown=3
doc_id: doc_00035045
file_path: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/raw/kaggle_extracted/test/invoice/506868459_506868460.tif
OCR snippet:
24.89 02:48 pm 7 pot promotional marketing tnc vad weaehuaor street ,enieago,iulingi coeld.f1einas onde % qvotatton for mr. jim bowling datel 1/24/89 field marketing menager . . reynolds tobaceo company job mor 07978- 401 , mein street wineton-salem, nc 27102 proteot: camel dpring rasort program execution description: yor program management and execution of tha camel spring resort program in daytena beach, piorida from march 12, 1989 through march 25, 1989 including the following: . sameoth movag execution . performance pay for snooth moves, emcee, equipment (sound and lights), technicians, assistants, and gostumed camel for tvo weeks. . . . $28,800, , -expensex for the above including van rental, limo rental, per dien, hotel and air

## Save model artifacts

For reproducibility and shared-team comparison, we save:
- the fitted vectorizer
- the selected best model
- metrics
- predictions
- confusion matrices
- top feature tables

In [14]:
# =========================
# 13. Save model bundle
# =========================
model_bundle_path = MODELS_DIR / "ocr_tfidf_baseline_bundle.joblib"
save_model_bundle(
    save_path=model_bundle_path,
    model=best_model,
    vectorizer=vectorizer,
    metadata={
        "seed": SEED,
        "target_classes": TARGET_CLASSES,
        "selected_model_name": best_run["model_name"],
        "selected_params": best_run["params"],
        "validation_metrics": best_run["val_metrics"],
        "test_metrics": test_metrics,
    }
)
print("Saved model bundle to:", model_bundle_path)

Saved model bundle to: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/ocr_tfidf_baseline_bundle.joblib


In [15]:
from __future__ import annotations

import re
from typing import Iterable, Optional, Tuple

from scipy.sparse import hstack, csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer


WHITESPACE_RE = re.compile(r"\s+")
CONTROL_RE = re.compile(r"[\x00-\x08\x0B-\x1F\x7F]")
BROKEN_SYMBOL_RE = re.compile(r"[^\w\s\.,:;/%$€£¥#@&\-\(\)/]")


def normalize_whitespace(text: str) -> str:
    text = WHITESPACE_RE.sub(" ", text).strip()
    return text


def basic_ocr_junk_cleanup(text: str) -> str:
    """
    Conservative cleanup:
    - remove control chars
    - keep currency symbols, punctuation that may matter for dates / invoice IDs
    """
    text = CONTROL_RE.sub(" ", text)
    text = BROKEN_SYMBOL_RE.sub(" ", text)
    text = normalize_whitespace(text)
    return text


def clean_ocr_text(text: Optional[str]) -> str:
    """
    Conservative OCR normalization:
    - lowercase
    - whitespace normalization
    - preserve useful invoice symbols
    - avoid destructive stripping of dates/currency/IDs
    """
    if text is None:
        return ""

    text = str(text)
    text = text.lower()
    text = basic_ocr_junk_cleanup(text)

    # Normalize repeated punctuation a little, but keep useful separators
    text = re.sub(r"([:;/,\.\-]){2,}", r"\1", text)

    # Remove extreme single-character OCR bursts except useful symbols
    text = re.sub(r"\b[a-z]{1}\b", " ", text)

    text = normalize_whitespace(text)
    return text


def contains_currency_like_pattern(text: str) -> bool:
    return bool(re.search(r"[$€£¥]\s?\d|\d[\d,\.]*\s?[$€£¥]", text))


def contains_date_like_pattern(text: str) -> bool:
    patterns = [
        r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
        r"\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b",
        r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s+\d{1,2},?\s+\d{2,4}\b",
    ]
    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)


def build_word_tfidf_vectorizer(
    ngram_range: Tuple[int, int] = (1, 2),
    max_features: int = 30000,
    min_df: int = 2,
    max_df: float = 0.98,
) -> TfidfVectorizer:
    return TfidfVectorizer(
        analyzer="word",
        ngram_range=ngram_range,
        max_features=max_features,
        min_df=min_df,
        max_df=max_df,
        strip_accents="unicode",
        sublinear_tf=True,
    )


def build_char_tfidf_vectorizer(
    ngram_range: Tuple[int, int] = (3, 5),
    max_features: int = 30000,
    min_df: int = 2,
    max_df: float = 1.0,
) -> TfidfVectorizer:
    return TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=ngram_range,
        max_features=max_features,
        min_df=min_df,
        max_df=max_df,
        sublinear_tf=True,
    )


class CombinedWordCharTfidfVectorizer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        word_ngram_range: Tuple[int, int] = (1, 2),
        char_ngram_range: Tuple[int, int] = (3, 5),
        max_word_features: int = 30000,
        max_char_features: int = 30000,
        min_df: int = 2,
    ) -> None:
        self.word_ngram_range = word_ngram_range
        self.char_ngram_range = char_ngram_range
        self.max_word_features = max_word_features
        self.max_char_features = max_char_features
        self.min_df = min_df

        self.word_vectorizer = build_word_tfidf_vectorizer(
            ngram_range=word_ngram_range,
            max_features=max_word_features,
            min_df=min_df,
        )
        self.char_vectorizer = build_char_tfidf_vectorizer(
            ngram_range=char_ngram_range,
            max_features=max_char_features,
            min_df=min_df,
        )

    def fit(self, X, y=None):
        self.word_vectorizer.fit(X)
        self.char_vectorizer.fit(X)
        return self

    def transform(self, X):
        X_word = self.word_vectorizer.transform(X)
        X_char = self.char_vectorizer.transform(X)
        return hstack([X_word, X_char]).tocsr()

    def fit_transform(self, X, y=None):
        X_word = self.word_vectorizer.fit_transform(X)
        X_char = self.char_vectorizer.fit_transform(X)
        return hstack([X_word, X_char]).tocsr()

    def get_feature_names_out(self):
        word_names = [f"WORD::{x}" for x in self.word_vectorizer.get_feature_names_out()]
        char_names = [f"CHAR::{x}" for x in self.char_vectorizer.get_feature_names_out()]
        return word_names + char_names


def build_combined_word_char_vectorizer(
    word_ngram_range: Tuple[int, int] = (1, 2),
    char_ngram_range: Tuple[int, int] = (3, 5),
    max_word_features: int = 30000,
    max_char_features: int = 30000,
    min_df: int = 2,
) -> CombinedWordCharTfidfVectorizer:
    return CombinedWordCharTfidfVectorizer(
        word_ngram_range=word_ngram_range,
        char_ngram_range=char_ngram_range,
        max_word_features=max_word_features,
        max_char_features=max_char_features,
        min_df=min_df,
    )

In [16]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, recall_score
from sklearn.svm import LinearSVC


def train_logistic_regression_baseline(
    X_train,
    y_train,
    C: float = 1.0,
    random_state: int = 42,
) -> LogisticRegression:
    model = LogisticRegression(
        C=C,
        max_iter=4000,
        multi_class="ovr",
        solver="liblinear",
        random_state=random_state,
    )
    model.fit(X_train, y_train)
    return model


def train_linear_svm_baseline(
    X_train,
    y_train,
    C: float = 1.0,
    random_state: int = 42,
) -> LinearSVC:
    model = LinearSVC(
        C=C,
        random_state=random_state,
    )
    model.fit(X_train, y_train)
    return model


def _compute_metrics(y_true, y_pred, labels: List[str]) -> Dict[str, Any]:
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")

    p, r, f1, s = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0
    )

    per_class = {}
    for cls, pp, rr, ff, ss in zip(labels, p, r, f1, s):
        per_class[cls] = {
            "precision": float(pp),
            "recall": float(rr),
            "f1": float(ff),
            "support": int(ss),
        }

    invoice_recall = per_class["invoice"]["recall"] if "invoice" in per_class else None

    return {
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "invoice_recall": invoice_recall,
        "per_class": per_class,
    }


def evaluate_classifier(model, X, y_true, label_order: List[str]):
    y_pred = model.predict(X)
    metrics = _compute_metrics(y_true, y_pred, labels=label_order)
    return metrics, y_pred


def select_best_model_from_validation(
    candidate_runs: List[Dict[str, Any]],
    primary_metric: str = "macro_f1",
) -> Dict[str, Any]:
    best = None
    best_score = -np.inf

    for run in candidate_runs:
        score = run["val_metrics"][primary_metric]
        if score > best_score:
            best_score = score
            best = run
        elif score == best_score:
            # tie-breaker: invoice recall, then accuracy
            current_ir = run["val_metrics"].get("invoice_recall", -1)
            best_ir = best["val_metrics"].get("invoice_recall", -1)

            if current_ir > best_ir:
                best = run
            elif current_ir == best_ir:
                current_acc = run["val_metrics"].get("accuracy", -1)
                best_acc = best["val_metrics"].get("accuracy", -1)
                if current_acc > best_acc:
                    best = run

    if best is None:
        raise ValueError("No candidate runs were provided.")
    return best


def extract_top_features_per_class(model, vectorizer, top_k: int = 20) -> pd.DataFrame:
    if not hasattr(model, "coef_"):
        raise ValueError("Model does not expose linear coefficients.")

    feature_names = np.array(vectorizer.get_feature_names_out())
    classes = list(model.classes_)
    rows = []

    for class_idx, cls in enumerate(classes):
        coef = model.coef_[class_idx]
        top_pos_idx = np.argsort(coef)[-top_k:][::-1]

        for rank, feat_idx in enumerate(top_pos_idx, start=1):
            rows.append({
                "class_name": cls,
                "rank": rank,
                "feature": feature_names[feat_idx],
                "weight": float(coef[feat_idx]),
            })

    return pd.DataFrame(rows)


def save_model_bundle(save_path: Path, model, vectorizer, metadata: Dict[str, Any]) -> None:
    save_path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "model": model,
        "vectorizer": vectorizer,
        "metadata": metadata,
    }
    joblib.dump(payload, save_path)


def load_model_bundle(load_path: Path) -> Dict[str, Any]:
    return joblib.load(load_path)

Minimal Expected Interfaces for Reused Modules

In [17]:
from __future__ import annotations

from pathlib import Path
from typing import Optional

import pandas as pd

def tesseract_is_available() -> bool:
    """
    Return True if pytesseract and the tesseract binary are available.
    """

def ensure_ocr_for_split(
    df: pd.DataFrame,
    split_name: str,
    cache_dir: Path,
    force_recompute: bool = False,
) -> pd.DataFrame:
    """
    Input:
        df with columns: doc_id, file_path, class_name
    Output:
        same df with an added column: ocr_text

    Behavior:
    - load cached OCR if present
    - otherwise run OCR and cache it
    - never silently fail
    - unreadable files should produce empty text plus a warning/log
    """

def load_cached_ocr_for_split(
    split_name: str,
    cache_dir: Path,
) -> Optional[pd.DataFrame]:
    """
    Returns cached OCR dataframe if available, else None.
    """

In [18]:
from __future__ import annotations

from pathlib import Path
from typing import List

import json
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

def plot_confusion_matrix_matplotlib(
    y_true,
    y_pred,
    labels: List[str],
    title: str,
    save_path: Path,
) -> None:
    """
    Save a confusion matrix figure using matplotlib only.
    """

def save_classification_report_table(
    y_true,
    y_pred,
    save_path: Path,
    labels: List[str],
) -> pd.DataFrame:
    """
    Save the classification report to CSV.
    """

def save_metrics_json(metrics: dict, save_path: Path) -> None:
    """
    Save a metrics dictionary as JSON.
    """

Fallback Implementation:

In [19]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

def plot_confusion_matrix_matplotlib(y_true, y_pred, labels, title, save_path):
    save_path.parent.mkdir(parents=True, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(8, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

def save_classification_report_table(y_true, y_pred, save_path, labels):
    save_path.parent.mkdir(parents=True, exist_ok=True)
    report = classification_report(
        y_true, y_pred, labels=labels, output_dict=True, zero_division=0
    )
    df = pd.DataFrame(report).T
    df.to_csv(save_path)
    return df

def save_metrics_json(metrics, save_path):
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

Conclusion:

- Verified the label mismatch issue programmatically
- Built the OCR text loading/caching workflow
- Implemented conservative OCR-aware text cleaning
- Built TF-IDF features using word and character n-grams
- Trained and compared Linear SVM and Logistic Regression
- Selected the best model using validation only
- Evaluated final test performance once
- Saved predictions, metrics, figures, and model bundle
- Added feature interpretation and error analysis